# 09 — Series temporales

Trabajo con fechas, agrupaciones por período, ventanas móviles y métricas de cambio en el tiempo.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d/%m/%Y')
print(df['Order Date'].dtype)
print(df['Order Date'].describe())


## Accessor dt — extraer componentes de fecha

In [ ]:
# Todos los atributos disponibles en .dt para columnas datetime
print(df['Order Date'].dt.year.value_counts().sort_index())
print()
print(df['Order Date'].dt.month.value_counts().sort_index())
print()
print('Día de semana (0=lunes):')
print(df['Order Date'].dt.dayofweek.value_counts().sort_index())


In [ ]:
# Duración entre fechas
df['dias_envio'] = (df['Ship Date'] - df['Order Date']).dt.days
print(df['dias_envio'].describe().round(1))
print()
print('Distribución días de envío:')
print(df['dias_envio'].value_counts().sort_index().head(10))


## Agrupar por período con to_period()

`to_period('M')` convierte una fecha a su período mensual (2024-11). Permite agrupar por mes manteniendo el año — a diferencia de `dt.month` que agrupa todos los eneros juntos.

In [ ]:
# Agrupar por mes conservando el año
ventas_mes = (
    df.groupby(df['Order Date'].dt.to_period('M'))['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Order Date': 'periodo', 'Sales': 'revenue'})
)
ventas_mes['periodo'] = ventas_mes['periodo'].astype(str)
print(ventas_mes.tail(12))


In [ ]:
# Agrupar por trimestre
ventas_q = (
    df.groupby(df['Order Date'].dt.to_period('Q'))['Sales']
    .agg(revenue='sum', pedidos='count')
    .reset_index()
)
ventas_q['Order Date'] = ventas_q['Order Date'].astype(str)
print(ventas_q)


## resample() — agrupación temporal con DatetimeIndex

`resample` es más potente que `groupby(to_period)` para series temporales: permite frecuencias como `W` (semana), `BM` (fin de mes laboral), etc. Requiere que el índice sea `DatetimeIndex`.

In [ ]:
# Establecer Order Date como índice para usar resample
df_idx = df.set_index('Order Date').sort_index()

# Agregar por mes
por_mes = df_idx['Sales'].resample('ME').sum()   # ME = Month End
print(por_mes.tail(12))
print()

# Agregar por semana
por_semana = df_idx['Sales'].resample('W').sum()
print(f'Semanas en el dataset: {len(por_semana)}')


## rolling() — ventana móvil

Calcula estadísticas sobre una ventana de N períodos consecutivos. Útil para suavizar series ruidosas y detectar tendencias.

In [ ]:
por_mes = df_idx['Sales'].resample('ME').sum().reset_index()
por_mes.columns = ['fecha', 'revenue']

# Media móvil de 3 meses
por_mes['media_movil_3m'] = por_mes['revenue'].rolling(window=3).mean().round(0)

# Los primeros N-1 valores son NaN porque no hay suficientes períodos anteriores
print(por_mes[['fecha', 'revenue', 'media_movil_3m']].tail(12))


## pct_change(), diff() y shift()

In [ ]:
# pct_change() — variación porcentual respecto al período anterior
por_mes['variacion_pct'] = por_mes['revenue'].pct_change().mul(100).round(1)

# diff() — diferencia absoluta respecto al período anterior
por_mes['variacion_abs'] = por_mes['revenue'].diff().round(0)

# shift(1) — desplaza la serie N posiciones (valor del período anterior)
por_mes['revenue_mes_anterior'] = por_mes['revenue'].shift(1)

print(por_mes[['fecha', 'revenue', 'variacion_pct', 'revenue_mes_anterior']].tail(8))


---
## Resumen

| Operación | Sintaxis |
|-----------|----------|
| Componentes de fecha | `df['col'].dt.year`, `.dt.month`, `.dt.dayofweek` |
| Duración | `(fecha2 - fecha1).dt.days` |
| Agrupar por mes+año | `df.groupby(df['col'].dt.to_period('M'))` |
| Resample por frecuencia | `df.set_index('fecha').resample('ME').sum()` |
| Ventana móvil | `.rolling(3).mean()` |
| Variación porcentual | `.pct_change()` |
| Diferencia absoluta | `.diff()` |
| Período anterior | `.shift(1)` |
